# Data Cleaning

In this notebook we explain how we prepare our data before the visual graph creation

In [1]:
import pandas as pd
from pathlib import Path

In [2]:
input_file = Path("simpsons_episodes.csv")
output_file = Path("simpsons_episodes_clean.csv")

In [3]:
df = pd.read_csv(input_file)
df.head()

,id,image_url,imdb_rating,imdb_votes,number_in_season,number_in_series,original_air_date,original_air_year,production_code,season,title,us_viewers_in_millions,video_url,views
0,10,http://static-media.fxx.com/img/FX_Networks_-_...,7.4,1511.0,10,10,1990-03-25,1990,7G10,1,Homer's Night Out,30.3,http://www.simpsonsworld.com/video/275197507879,50816.0
1,12,http://static-media.fxx.com/img/FX_Networks_-_...,8.3,1716.0,12,12,1990-04-29,1990,7G12,1,Krusty Gets Busted,30.4,http://www.simpsonsworld.com/video/288019523914,62561.0
2,14,http://static-media.fxx.com/img/FX_Networks_-_...,8.2,1638.0,1,14,1990-10-11,1990,7F03,2,"Bart Gets an ""F""",33.6,http://www.simpsonsworld.com/video/260539459671,59575.0
3,17,http://static-media.fxx.com/img/FX_Networks_-_...,8.1,1457.0,4,17,1990-11-01,1990,7F01,2,Two Cars in Every Garage and Three Eyes on Eve...,26.1,http://www.simpsonsworld.com/video/260537411822,64959.0
4,19,http://static-media.fxx.com/img/FX_Networks_-_...,8.0,1366.0,6,19,1990-11-15,1990,7F08,2,Dead Putting Society,25.4,http://www.simpsonsworld.com/video/260539459670,50691.0


## Columns Selection

In [4]:
important_cols = [
    "title",
    "season",
    "number_in_season",
    "number_in_series",
    "original_air_date",
    "imdb_rating",
    "imdb_votes",
    "us_viewers_in_millions",
]

df = df[important_cols].copy()
df.head()

,title,season,number_in_season,number_in_series,original_air_date,imdb_rating,imdb_votes,us_viewers_in_millions
0,Homer's Night Out,1,10,10,1990-03-25,7.4,1511.0,30.3
1,Krusty Gets Busted,1,12,12,1990-04-29,8.3,1716.0,30.4
2,"Bart Gets an ""F""",2,1,14,1990-10-11,8.2,1638.0,33.6
3,Two Cars in Every Garage and Three Eyes on Eve...,2,4,17,1990-11-01,8.1,1457.0,26.1
4,Dead Putting Society,2,6,19,1990-11-15,8.0,1366.0,25.4


## Type conversion

In [5]:
print("Types before conversion:")
print(df.dtypes)

df["original_air_date"] = pd.to_datetime(df["original_air_date"], errors="coerce")

print("Types after conversion:")
print(df.dtypes)

Types before conversion:
title                      object
season                      int64
number_in_season            int64
number_in_series            int64
original_air_date          object
imdb_rating               float64
imdb_votes                float64
us_viewers_in_millions    float64
dtype: object
Types after conversion:
title                             object
season                             int64
number_in_season                   int64
number_in_series                   int64
original_air_date         datetime64[ns]
imdb_rating                      float64
imdb_votes                       float64
us_viewers_in_millions           float64
dtype: object


## NA value check

In [6]:
print("NA's per column:")
print(df.isna().sum())

print("Rows with at least one NA value:")
df[df.isna().any(axis=1)]

NA's per column:
title                     0
season                    0
number_in_season          0
number_in_series          0
original_air_date         0
imdb_rating               3
imdb_votes                3
us_viewers_in_millions    6
dtype: int64
Rows with at least one NA value:


,title,season,number_in_season,number_in_series,original_air_date,imdb_rating,imdb_votes,us_viewers_in_millions
59,Lisa's Date with Density,8,7,160,1996-12-15,7.8,1005.0,NaN
65,The Canine Mutiny,8,20,173,1997-04-13,7.7,913.0,NaN
234,"Friends and Family""[203]",28,2,598,2016-10-02,NaN,NaN,NaN
235,"The Town""[205]",28,3,599,2016-10-09,NaN,NaN,NaN
236,"Treehouse of Horror XXVII""[207]",28,4,600,2016-10-16,NaN,NaN,NaN
320,Hurricane Neddy,8,8,161,1996-12-29,8.8,1268.0,NaN


## Imputation and Filtering

In [7]:
df = df[df["season"] != 28].copy()

df["us_viewers_in_millions"] = (
    df.groupby("season")["us_viewers_in_millions"]
    .transform(lambda s: s.interpolate())
)

df.head()

,title,season,number_in_season,number_in_series,original_air_date,imdb_rating,imdb_votes,us_viewers_in_millions
0,Homer's Night Out,1,10,10,1990-03-25,7.4,1511.0,30.3
1,Krusty Gets Busted,1,12,12,1990-04-29,8.3,1716.0,30.4
2,"Bart Gets an ""F""",2,1,14,1990-10-11,8.2,1638.0,33.6
3,Two Cars in Every Garage and Three Eyes on Eve...,2,4,17,1990-11-01,8.1,1457.0,26.1
4,Dead Putting Society,2,6,19,1990-11-15,8.0,1366.0,25.4


## Duplicate check

In [8]:
duplicates = df[df.duplicated()]
print("Duplicates found:", len(duplicates))
duplicates

Duplicates found: 0


,title,season,number_in_season,number_in_series,original_air_date,imdb_rating,imdb_votes,us_viewers_in_millions


## Derivated Columns

In [9]:
# 1. Separate the date into individual columns
df["air_year"] = df["original_air_date"].dt.year
df["air_month"] = df["original_air_date"].dt.month
df["weekday_num"] = df["original_air_date"].dt.weekday

#2. Join season and number_in_season
df["season_episode"] = (
    "S"
    + df["season"].astype(int).astype(str).str.zfill(2)
    + "E"
    + df["number_in_season"].astype(int).astype(str).str.zfill(2)
)

# 3. Number of episodes per season within the dataset
episodes_per_season = df.groupby("season")["number_in_season"].count()
df["episodes_in_season_dataset"] = df["season"].map(episodes_per_season)

df.head()

,title,season,number_in_season,number_in_series,original_air_date,imdb_rating,imdb_votes,us_viewers_in_millions,air_year,air_month,weekday_num,season_episode,episodes_in_season_dataset
0,Homer's Night Out,1,10,10,1990-03-25,7.4,1511.0,30.3,1990,3,6,S01E10,13
1,Krusty Gets Busted,1,12,12,1990-04-29,8.3,1716.0,30.4,1990,4,6,S01E12,13
2,"Bart Gets an ""F""",2,1,14,1990-10-11,8.2,1638.0,33.6,1990,10,3,S02E01,22
3,Two Cars in Every Garage and Three Eyes on Eve...,2,4,17,1990-11-01,8.1,1457.0,26.1,1990,11,3,S02E04,22
4,Dead Putting Society,2,6,19,1990-11-15,8.0,1366.0,25.4,1990,11,3,S02E06,22


## Ordering

In [10]:
df = df.sort_values(
    ["original_air_date", "season", "number_in_season"]
).reset_index(drop=True)

df

,title,season,number_in_season,number_in_series,original_air_date,imdb_rating,imdb_votes,us_viewers_in_millions,air_year,air_month,weekday_num,season_episode,episodes_in_season_dataset
0,Simpsons Roasting on an Open Fire,1,1,1,1989-12-17,8.2,3734.0,26.70,1989,12,6,S01E01,13
1,Bart the Genius,1,2,2,1990-01-14,7.8,1973.0,24.50,1990,1,6,S01E02,13
2,Homer's Odyssey,1,3,3,1990-01-21,7.5,1709.0,27.50,1990,1,6,S01E03,13
3,There's No Disgrace Like Home,1,4,4,1990-01-28,7.8,1701.0,20.20,1990,1,6,S01E04,13
4,Bart the General,1,5,5,1990-02-04,8.1,1732.0,27.10,1990,2,6,S01E05,13
...,...,...,...,...,...,...,...,...,...,...,...,...,...
591,How Lisa Got Her Marge Back,27,18,592,2016-04-10,6.4,228.0,2.55,2016,4,6,S27E18,22
592,Fland Canyon,27,19,593,2016-04-24,7.1,229.0,2.77,2016,4,6,S27E19,22
593,To Courier with Love,27,20,594,2016-05-08,6.7,193.0,2.52,2016,5,6,S27E20,22
594,Simprovised,27,21,595,2016-05-15,6.4,227.0,2.80,2016,5,6,S27E21,22


## Export clean CSV

In [11]:
df.to_csv(output_file, index=False)
print(f"File saved in: {output_file}")

File saved in: simpsons_episodes_clean.csv
